In [49]:
import os
import numpy as np
import pandas as pd
import joblib

np.random.seed(42)

PROCESSED_DIR = r"C:\Project_EquiGuard\data\processed"
MODELS_DIR = r"C:\Project_EquiGuard\outputs\models"

os.makedirs(MODELS_DIR, exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


In [50]:
# Load all three files

train = pd.read_csv(os.path.join(PROCESSED_DIR, "train_raw.csv"))
val   = pd.read_csv(os.path.join(PROCESSED_DIR, "val_raw.csv"))
test  = pd.read_csv(os.path.join(PROCESSED_DIR, "test_raw.csv"))

print("Files loaded.")
print(f"Train: {train.shape}")
print(f"Val:   {val.shape}")
print(f"Test:  {test.shape}")

Files loaded.
Train: (210, 36)
Val:   (132, 35)
Test:  (133, 35)


In [51]:
# Drop non-feature columns

DROP_COLS = [
    "horseID", "HorseID", "STUDY", "Age (years)", "Mass (Kg)",
    "Breed", "Sex", "YOB", "LL", "WH", "Shoe_F", "Shoe_H",
    "Ground", "Location", "Date", "Runway_length", "Placer",
    "Handler", "Whipper", "Speed_mes", "source", "Exclude",
    "Max value", "label"
]

# Save labels before dropping
y_train = train["label"].values
y_val   = val["label"].values
y_test  = test["label"].values

# Remove metadata and non-feature columnstrain = train.drop(columns=[c for c in DROP_COLS if c in train.columns])
val   = val.drop(columns=[c for c in DROP_COLS if c in val.columns])
test  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

print(f"Train features: {train.shape[1]}")
print(f"Val features:   {val.shape[1]}")
print(f"Test features:  {test.shape[1]}")

Train features: 28
Val features:   28
Test features:  28


In [52]:
# Missing value handling
# Fit medians on numeric columns of train only
# Select numeric columns only before computing median
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

train_medians = train[numeric_cols].median()

train = train.fillna(train_medians)
val   = val.fillna(train_medians)
test  = test.fillna(train_medians)

# Drop any remaining non-numeric columns
train = train[numeric_cols]
val   = val[[c for c in numeric_cols if c in val.columns]]
test  = test[[c for c in numeric_cols if c in test.columns]]

print(f"Numeric columns kept: {len(numeric_cols)}")
print(f"Nulls in train: {train.isnull().sum().sum()}")
print(f"Nulls in val:   {val.isnull().sum().sum()}")
print(f"Nulls in test:  {test.isnull().sum().sum()}")

Numeric columns kept: 28
Nulls in train: 0
Nulls in val:   0
Nulls in test:  0


In [53]:
# Align columns
# Keep only columns that exist in ALL THREE files

shared_cols = list(
    set(train.columns) & set(val.columns) & set(test.columns)
)

train = train[shared_cols]
val   = val[shared_cols]
test  = test[shared_cols]

print(f"Shared features: {len(shared_cols)}")
print(shared_cols)

Shared features: 28
['HHDisRHasym', 'MnDisRHasym', 'UpDisHsym', 'MnDisLHasym', 'UpDisLFasym', 'MxDisLHasym', 'UpDisLHasym', 'HHD', 'UpDSac', 'HHDisLHasym', 'MnDisHsym', 'MnDisRFasym', 'MxDHead', 'UpDHead', 'MnDisFsym', 'MnDisLFasym', 'MxDSac', 'MxDisRFasym', 'MxDisHsym', 'UpDisFsym', 'MnDHead', 'HHDisHsym', 'MnDSac', 'MxDisLFasym', 'MxDisRHasym', 'MxDisFsym', 'UpDisRFasym', 'UpDisRHasym']


In [54]:
# Build feature matrices
X_train = train.values
X_val   = val.values
X_test  = test.values

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

print(f"\ny_train distribution: {np.unique(y_train, return_counts=True)}")
print(f"y_val distribution:   {np.unique(y_val, return_counts=True)}")
print(f"y_test distribution:  {np.unique(y_test, return_counts=True)}")

X_train: (210, 28) | y_train: (210,)
X_val:   (132, 28)   | y_val:   (132,)
X_test:  (133, 28)  | y_test:  (133,)

y_train distribution: (array([0, 1]), array([ 42, 168]))
y_val distribution:   (array([0, 1]), array([ 10, 122]))
y_test distribution:  (array([0, 1]), array([ 10, 123]))


In [55]:
# Save processed arrays and metadata
ARRAYS_DIR = os.path.join(PROCESSED_DIR, "arrays")
os.makedirs(ARRAYS_DIR, exist_ok=True)

np.save(os.path.join(ARRAYS_DIR, "X_train.npy"), X_train)
np.save(os.path.join(ARRAYS_DIR, "y_train.npy"), y_train)
np.save(os.path.join(ARRAYS_DIR, "X_val.npy"),   X_val)
np.save(os.path.join(ARRAYS_DIR, "y_val.npy"),   y_val)
np.save(os.path.join(ARRAYS_DIR, "X_test.npy"),  X_test)
np.save(os.path.join(ARRAYS_DIR, "y_test.npy"),  y_test)

# Save shared column names for reference later
joblib.dump(shared_cols, os.path.join(MODELS_DIR, "feature_names.pkl"))

# Save medians for any future inference
joblib.dump(train_medians, os.path.join(MODELS_DIR, "train_medians.pkl"))

print("All arrays saved.")
print(f"Location: {ARRAYS_DIR}")
print(f"Feature names saved to: {MODELS_DIR}/feature_names.pkl")

All arrays saved.
Location: C:\Project_EquiGuard\data\processed\arrays
Feature names saved to: C:\Project_EquiGuard\outputs\models/feature_names.pkl
